In [47]:
import pandas as pd
import numpy as np

# Giả sử 'df_raw' là dataframe chứa 31 thuộc tính ban đầu của bạn
# Load dataset

DATA_PATH = "../../data/processed/phone_specs_preprocessed.csv"

df = pd.read_csv(DATA_PATH)
df = df[df['phone_type'] != 'feature_phone']
display(df.describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
price_vnd,395.0,14136405.06,11796979.00,1590000.0,4740000.0,8990000.0,22690000.00,63990000.00
os_version,316.0,14.39,3.23,8.1,13.0,14.0,15.00,26.00
screen_size_inch,393.0,6.64,0.32,5.4,6.5,6.7,6.78,8.12
refresh_rate_hz,361.0,109.88,21.70,60.0,90.0,120.0,120.00,165.00
resolution_width,361.0,1118.61,258.65,720.0,1080.0,1080.0,1256.00,2268.00
resolution_height,361.0,2392.09,447.49,1440.0,2340.0,2400.0,2712.00,3216.00
resolution_total_pixels,361.0,2764687.20,996660.90,1036800.0,2527200.0,2604960.0,3437280.00,5533920.00
ram_gb,364.0,8.30,3.27,2.0,6.0,8.0,12.00,16.00
storage_gb,393.0,273.34,231.74,16.0,128.0,256.0,256.00,2048.00
battery_mah,343.0,4976.99,1016.60,323.0,4500.0,5000.0,5200.00,8300.00


## Rating màn hình

In [48]:
df['refresh_rate_hz'] = df['refresh_rate_hz'].fillna(60)

In [49]:
df =df[df['product_name']!='Điện thoại OPPO']
df['screen_size_fixed'] = df.groupby('price_segment')['screen_size_inch'].transform(lambda x: x.fillna(x.median()))

# Bạn cũng nên làm tương tự cho độ phân giải (nếu có giá trị thiếu)
df['resolution_total_pixels'] = df.groupby('price_segment')['resolution_total_pixels'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
)
features = ['screen_size_fixed', 'panel_group', 'resolution_total_pixels']
display(df[features].isna().sum())
print()

screen_size_fixed          0
panel_group                0
resolution_total_pixels    0
dtype: int64

In [50]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Giả lập tập dữ liệu lớn hơn một chút để K-Means hoạt động ổn định
# data = {
#     'screen_size': [15.6, 14.0, 27.0, 13.3, 15.6, 11.6, 32.0, 14.0],
#     'panel_group': ['TN', 'IPS', 'IPS', 'OLED', 'VA', 'TN', 'Mini-LED', 'OLED'],
#     'resolution_total_pixels': [2073600, 2073600, 8294400, 5184000, 2073600, 1049088, 8294400, 3686400]
# }
# df = pd.DataFrame(data)

# 2. Xử lý thuộc tính chữ (Ordinal Encoding dựa trên thứ tự chất lượng)
panel_map = {'Unknown':0, 'LCD': 2, 'AMOLED': 3}
df['panel_score'] = df['panel_group'].map(panel_map)

#Chọn các tính năng để đưa vào K-Means
features = ['screen_size_fixed', 'panel_score', 'resolution_total_pixels']
X = df[features]

# 3. CHUẨN HÓA DỮ LIỆU (Bắt buộc phải có với K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. CHẠY K-MEANS (Chia làm 4 cụm tương ứng 4 Tiers)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster_raw'] = kmeans.fit_predict(X_scaled)

# 5. ĐỊNH DANH LẠI THỨ TỰ (Sắp xếp cụm từ thấp đến cao theo độ phân giải trung bình)
# Tính độ phân giải trung bình của từng cụm thô
cluster_order = df.groupby('cluster_raw')['resolution_total_pixels'].mean().sort_values().index

# Tạo bản đồ ánh xạ từ cụm ngẫu nhiên sang tên Tier chuẩn
tier_names = [1,2,3,4]
cluster_mapping = {raw_cluster: tier_names[i] for i, raw_cluster in enumerate(cluster_order)}

# Gán nhãn cuối cùng
df['display_tier'] = df['cluster_raw'].map(cluster_mapping)

# Xem kết quả kiểm chứng
print(df[df['price_segment']=='flagship'][['screen_size_inch', 'panel_group', 'resolution_height', 'display_tier']])

     screen_size_inch panel_group  resolution_height  display_tier
4                6.20      AMOLED             3200.0             2
10               8.12      AMOLED                NaN             4
12               6.30      AMOLED             2340.0             2
18               6.90      AMOLED             3200.0             3
20               6.10      AMOLED             2532.0             2
..                ...         ...                ...           ...
398              6.60      AMOLED             2340.0             3
402              8.00      AMOLED             2184.0             4
405              7.60      AMOLED             2176.0             4
408              6.70      AMOLED             2778.0             3
410              7.60      AMOLED             2176.0             4

[112 rows x 4 columns]


## Rating chip

In [51]:
print(df['chip_name'].head())

0           Helio G85 (12nm)
1         Snapdragon 6 Gen 3
3                 Exynos 850
4         Exynos 990 (7 nm+)
5    Qualcomm Snapdragon 685
Name: chip_name, dtype: str


In [52]:
import re
path_chip_rating = '../../data/raw/processor/processors_ratings.csv'
df_chip_rating = pd.read_csv(path_chip_rating)
# ---------------------------------------------------------
# 1. BƯỚC CHUẨN BỊ (Làm sạch cơ bản)
# ---------------------------------------------------------
# Chúng ta vẫn cần một chút làm sạch nhẹ để xóa các ký tự cản trở như ®, ™ và đưa về chữ thường
def light_clean(text):
    if pd.isna(text): 
        return ""
    text = str(text).lower()
    text = text.replace('(', '').replace(')', '')
    text = text.replace('+','')
    
    # 2. Xóa các ký tự thương hiệu như ®, ™
    text = re.sub(r'[®™]', '', text) 

    text = text.replace('thế hệ thứ', 'gen')
    text = text.replace('thế hệ', 'gen')

    
    
    # 3. Dọn dẹp khoảng trắng thừa (biến nhiều khoảng trắng liên tiếp thành 1 khoảng trắng)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def normalize_aliases(text):
    if '8 elite' in text:
        if 'gen 5' in text:
            return 'snapdragon 8 elite gen 5'
        else:
            # Ép tất cả các bản 8 elite còn lại (bản trơn, 3nm, for galaxy, gen 4...) về đúng tên chuẩn
            return 'snapdragon 8 elite gen 4'
    if 'apple a15' in text or text == 'a15': return 'a15 bionic'
    if 'chip a18' in text or text == 'a18': return 'apple a18'
    if 't612' in text: return 'tiger t612'
    if 't616' in text: return 'tiger t616'
    if 't606' in text: return 'tiger t606'
    if 'g99 ultimate' in text: return 'helio g99 ultimate'
    if 'g99' in text and 'helio' not in text: return 'helio g99'
    if 'g100' in text and 'helio' not in text: return 'helio g100'
    if '7025' in text and 'ultra' in text: return 'dimensity 7025 ultra'
    if '8895' in text: return 'exynos 8895'
    return text

# Áp dụng cho DataFrame chính
df['chip_clean'] = df['chip_name'].apply(light_clean).apply(normalize_aliases)

# Chuẩn hóa cột file CSV
df_chip_rating.columns = df_chip_rating.columns.str.lower()
df_chip_rating['processor_clean'] = df_chip_rating['processor'].apply(light_clean)
df_chip_rating['brand_clean'] = df_chip_rating['brand'].astype(str).str.lower().str.strip()
# ---------------------------------------------------------
# 2. TẠO TỪ ĐIỂN VÀ TÍNH ĐIỂM MIN TỰ ĐỘNG TỪ CỘT BRAND
# ---------------------------------------------------------
# Từ điển Chip
chip_df_grouped = df_chip_rating.groupby('processor_clean')['rating'].max().reset_index()
chip_dict = dict(zip(chip_df_grouped['processor_clean'], chip_df_grouped['rating']))
sorted_chip_names = sorted(chip_dict.keys(), key=len, reverse=True)

# Tự động tạo Dictionary lấy điểm Min của cả 8 hãng có trong file CSV
# Kết quả sẽ có dạng: {'qualcomm': 24, 'apple': 24, 'mediatek': 23, 'samsung': 23, ...}
brand_mins = df_chip_rating.groupby('brand_clean')['rating'].min().to_dict()

# ---------------------------------------------------------
# 3. QUÉT CHUỖI VÀ GÁN ĐIỂM (CẬP NHẬT 8 HÃNG)
# ---------------------------------------------------------
def get_final_chip_score(messy_name):
    if not messy_name or messy_name == 'unknown':
        return 0
        
    # 3.1: Dò khớp tên chip chính xác
    for clean_name in sorted_chip_names:
        if clean_name in messy_name:
            return chip_dict[clean_name]
            
    # 3.2: Dò từ khóa (Keywords) để nhận diện 8 hãng và giáng cấp xuống Min score
    if any(k in messy_name for k in ['qualcomm', 'snapdragon', 'sdm']): 
        return brand_mins.get('qualcomm', 0)
        
    if any(k in messy_name for k in ['apple', 'bionic', 'chip a']): 
        return brand_mins.get('apple', 0)
        
    if any(k in messy_name for k in ['mediatek', 'dimensity', 'helio', 'mt6', 'mt8']): 
        return brand_mins.get('mediatek', 0)
        
    if any(k in messy_name for k in ['samsung', 'exynos']): 
        return brand_mins.get('samsung', 0)
        
    if any(k in messy_name for k in ['xiaomi', 'surge', 'xring']): 
        return brand_mins.get('xiaomi', 0)
        
    if any(k in messy_name for k in ['hisilicon', 'kirin']): 
        return brand_mins.get('hisilicon', 0)
        
    if any(k in messy_name for k in ['google', 'tensor']): 
        return brand_mins.get('google', 0)
        
    if any(k in messy_name for k in ['unisoc', 'tiger', 'spreadtrum', 'ums', ' t6', ' t7']): 
        return brand_mins.get('unisoc', 0)
    
    # 3.3: Dữ liệu rác hoàn toàn (Ví dụ: "8 nhân", "Mali-G57")
    return 0

# ---------------------------------------------------------
# 4. THỰC THI VÀ DỌN DẸP
# ---------------------------------------------------------
df['chip_rating'] = df['chip_clean'].apply(get_final_chip_score)
df = df.drop(columns=['chip_clean'])

# Kiểm tra dữ liệu rớt đài
zero_score_chips = df[df['chip_rating'] == 0]['chip_name'].unique()
print("Dữ liệu rác (0 điểm):", zero_score_chips)

Dữ liệu rác (0 điểm): <StringArray>
[                                    '8 nhân',
 '10 nhân, 3.3 GHz, 2.74GHz, 2.36GHz, 1.8GHz',
                                    'Unknown',
                                      'T7200',
                                   'Mali-G57']
Length: 5, dtype: str


## KBRS

In [53]:
df['ip_status'] = df['ip_status'].astype(str).str.lower().str.strip()

df['ip_status'] = np.where(df['ip_status'] == 'certified', 1, 0)

print(df['ip_status'].unique())

[1 0]


In [54]:
# Bọc lót dữ liệu thiếu (NaN): Điền bằng số trung vị (50%) là 194.0g
df['weight_g'] = df['weight_g'].fillna(194.0)

df['weight_g'] = df['weight_g'].fillna(194.0)

# 2. Định nghĩa các điều kiện mốc cắt cho 2 nhóm đầu (Độ dài = 2)
conditions = [
    df['weight_g'] < 185.0,                               # Nhóm Nhẹ
    (df['weight_g'] >= 185.0) & (df['weight_g'] < 210.0)   # Nhóm Vừa
]

# 3. Định nghĩa nhãn tương ứng cho 2 điều kiện trên (Độ dài phải = 2)
# Điều kiện 1 ứng với nhãn 1, Điều kiện 2 ứng với nhãn 2
tier_labels = [1, 2] 

# 4. Tạo cột phân loại mới trong dataframe
# Những máy không thỏa mãn 2 điều kiện trên tự động nhận giá trị default là 3 (Nhóm Heavy)
df['weight_tier'] = np.select(conditions, tier_labels, default=3)

# Kiểm tra số lượng máy được phân bổ vào từng nhóm
print(df['weight_tier'].value_counts())

weight_tier
2    245
3     75
1     74
Name: count, dtype: int64


In [55]:
filter_features = ['product_name', 'price_vnd', 'price_segment', 'weight_tier', 'score_perf', 'score_cam', 'score_batt', 'score_disp', 'ip_status']

display_features = ['product_name', 'brand', 'url', 'price_vnd', 'price_segment',

       'promotion', 'os_family', 'os_version', 'chip_name',

       'chip_brand', 'screen_size_inch', 'panel',

       'refresh_rate_hz', 'resolution_width', 'resolution_height', 'ram_gb', 'storage_gb', 'battery_mah',

       'fast_charge_w', 'front_camera_mp', 'rear_main_camera_mp',

       'rear_camera_count', 'max_video_resolution', 'fps_at_max_resolution', 'weight_g', 'ip_rating'
]

Nhóm 1: Điều kiện cứng (Hard Filters - Bắt buộc thỏa mãn)

- Giá tiền tối đa.

- Phân khúc ưu tiên: Giá rẻ / Flagship cao cấp / Bất kỳ.

Nhóm 2: Mục đích sử dụng (Soft Constraints / Scoring - Dùng để xếp hạng)

- Liên lạc và cơ bản.

- Chơi game.

- Giải trí và Mạng xã hội.

- Chụp hình, quay phim.

- Pin trâu, sạc nhanh.

In [56]:
def calculate_feature_scores(df):
    """
    Hàm tổng hợp các biến vật lý thành 4 cột điểm (thang điểm 10) cho KBRS.
    """
    df_scored = df.copy()

    # Hàm Min-Max Scaling cục bộ (Đưa một cột số về khoảng 0.0 - 1.0)
    def min_max_scale(series):
        return (series - series.min()) / (series.max() - series.min() + 1e-9) # +1e-9 để tránh chia cho 0

    # ---------------------------------------------------------
    # 1. TÍNH ĐIỂM HIỆU NĂNG (score_perf)
    # Giả sử RAM chiếm 60% sức mạnh, Bộ nhớ trong chiếm 40%
    # ---------------------------------------------------------

    # Chuẩn hóa cả 3 tiêu chí về thang [0, 1]
    norm_chip = min_max_scale(df_scored['chip_rating'])
    norm_ram = min_max_scale(df_scored['ram_gb'])
    norm_storage = min_max_scale(df_scored['storage_gb'])

    # Tính điểm với trọng số: Chip (50%), RAM (30%), ROM (20%) - Nhân 10 để ra thang điểm 10
    df_scored['score_perf'] = ((norm_chip * 0.5) + (norm_ram * 0.3) + (norm_storage * 0.2)) * 10

# ---------------------------------------------------------
# 2. TÍNH ĐIỂM CAMERA (score_cam) THEO TRỌNG SỐ MỚI
# ---------------------------------------------------------
    norm_main_cam  = min_max_scale(df_scored['rear_main_camera_mp'])
    norm_front_cam = min_max_scale(df_scored['front_camera_mp'])
    norm_video_res = min_max_scale(df_scored['video_resolution_rank'])
    norm_fps       = min_max_scale(df_scored['fps_at_max_resolution'])
    norm_cam_count = min_max_scale(df_scored['rear_camera_count'])

    df_scored['score_cam'] = (
        (norm_main_cam  * 0.35) + 
        (norm_video_res * 0.25) + 
        (norm_front_cam * 0.20) + 
        (norm_fps       * 0.10) + 
        (norm_cam_count * 0.10)
    ) * 10

# ---------------------------------------------------------
# 3. TÍNH ĐIỂM PIN & SẠC (score_batt)
# Pin to chiếm 50%, Sạc nhanh chiếm 50%
# ---------------------------------------------------------
    norm_battery = min_max_scale(df_scored['battery_mah'])
    norm_charge = min_max_scale(df_scored['fast_charge_w'])
    
    df_scored['score_batt'] = ((norm_battery * 0.5) + (norm_charge * 0.5)) * 10

# ---------------------------------------------------------
# 4. TÍNH ĐIỂM MÀN HÌNH (score_disp)
# Tổng số pixel (độ nét) và Tần số quét (độ mượt)
# ---------------------------------------------------------
    norm_dis = min_max_scale(df_scored['display_tier'])
    norm_hz = min_max_scale(df_scored['refresh_rate_hz'])
    
    df_scored['score_disp'] = ((norm_dis * 0.6) + (norm_hz * 0.4)) * 10

    # Làm tròn điểm về 2 chữ số thập phân cho gọn
    score_cols = ['score_perf', 'score_cam', 'score_batt', 'score_disp']
    df_scored[score_cols] = df_scored[score_cols].round(2)

    return df_scored[filter_features]

In [58]:
# ==========================================
# 2. LÕI HỆ THỐNG KBRS (ENGINE)
# ==========================================
class KBRSEngine:
    def __init__(self, df_phones, mapping_matrix):
        self.df = df_phones.copy()
        self.mapping_matrix = mapping_matrix
        # Các cột điểm đã chuẩn hóa (0-10) dùng để nhân ma trận
        self.score_features = ['score_perf', 'score_cam', 'score_batt', 'score_disp']

    def recommend(self, user_needs, max_budget=None, top_k=3):
        """
        user_needs: Có thể truyền vào 1 chuỗi (String) hoặc 1 danh sách (List of Strings)
        """
        # 0. Chuẩn hóa đầu vào thành List để dễ xử lý
        if isinstance(user_needs, str):
            user_needs = [user_needs]
            
        # Lọc ra các nhu cầu hợp lệ có trong hệ thống
        valid_needs = [n for n in user_needs if n in self.mapping_matrix]
        if not valid_needs:
            return "Lỗi: Không tìm thấy nhu cầu hợp lệ nào."

        # BƯỚC 1: TÍNH TRUNG BÌNH TRỌNG SỐ CHO ĐA NHU CẦU
        combined_weights = {'perf': 0, 'cam': 0, 'batt': 0, 'disp': 0}
        
        for need in valid_needs:
            weights = self.mapping_matrix[need]['weights']
            for key in combined_weights:
                combined_weights[key] += weights.get(key, 0)
                
        # Chia trung bình để tổng trọng số không bị phình to
        num_needs = len(valid_needs)
        user_vector = np.array([
            combined_weights['perf'] / num_needs,
            combined_weights['cam'] / num_needs,
            combined_weights['batt'] / num_needs,
            combined_weights['disp'] / num_needs
        ])

        filtered_df = self.df.copy()

        # BƯỚC 2: GỘP LỌC CỨNG (Vòng lặp AND Logic)
        if max_budget:
            filtered_df = filtered_df[filtered_df['price_vnd'] <= max_budget]

        # Quét qua từng nhu cầu, máy nào không thỏa mãn luật cứng sẽ bị rớt đài dần
        for need in valid_needs:
            hard_filters = self.mapping_matrix[need].get('hard_filters', {})
            for col, condition in hard_filters.items():
                if col in filtered_df.columns:
                    filtered_df = filtered_df[condition(filtered_df[col])]

        # Nếu lọc cứng chọi nhau (Ví dụ: Vừa đòi Flagship vừa đòi Giá rẻ)
        if filtered_df.empty:
            return "Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI tất cả các nhu cầu bắt buộc của bạn."

        # BƯỚC 3: CHẤM ĐIỂM BẰNG MA TRẬN
        item_matrix = filtered_df[self.score_features].values
        filtered_df['total_score'] = np.dot(item_matrix, user_vector)

        # Xếp hạng
        final_result = filtered_df.sort_values(by='total_score', ascending=False).head(top_k)
        
        display_cols = ['product_name', 'price_vnd', ] + self.score_features + ['total_score']
        return final_result[display_cols].round(2)

In [ ]:
df_ready_for_kbrs = calculate_feature_scores(df)

In [67]:
df_ready_for_kbrs.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
price_vnd,394.0,14111395.94,11801489.25,1590000.00,4715000.00,8990000.00,22465000.00,63990000.00
weight_tier,394.0,2.00,0.62,1.00,2.00,2.00,2.00,3.00
score_perf,364.0,3.89,1.92,0.02,2.41,3.42,5.06,8.99
score_cam,279.0,3.45,1.36,0.86,2.48,3.32,4.16,9.03
score_batt,302.0,4.46,1.42,0.91,3.32,4.39,5.22,8.00
score_disp,394.0,4.23,2.43,0.00,2.07,4.29,6.29,8.29
ip_status,394.0,0.68,0.47,0.00,0.00,1.00,1.00,1.00


Mapping matrix

In [ ]:
PERF_THRESHOLD = 5.5 #Lấy mức lớn hơn 75%
CAM_THRESHOLD = 4.5

MAPPING_MATRIX_CONFIG = {
    'Gia_Re': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {'price_segment': lambda x: x == 'budget'}
    },
    'Flagship': {
        'weights': {'perf': 0.3, 'cam': 0.3, 'batt': 0.1, 'disp': 0.3},
        'hard_filters': {'price_segment': lambda x: x == 'flagship'}
    },
    'Lien_Lac_Co_Ban': {
        'weights': {'perf': 0.1, 'cam': 0.1, 'batt': 0.7, 'disp': 0.1},
        'hard_filters': {} # Không có điều kiện cứng
    },
    'Choi_Game': {
        'weights': {'perf': 0.7, 'cam': 0.05, 'batt': 0.15, 'disp': 0.1},
        'hard_filters': {
            'score_perf': lambda x: x >= PERF_THRESHOLD,      
            'refresh_rate_hz': lambda x: x >= 120.0, 
            'ram_gb': lambda x: x >= 8.0             
        }
    },
    'Giai_Tri_MXH': {
        'weights': {'perf': 0.2, 'cam': 0.1, 'batt': 0.3, 'disp': 0.4},
        'hard_filters': {'display_tier': lambda x: x >= 3}
    },
    'Chup_Hinh_Quay_Phim': {
        'weights': {'perf': 0.15, 'cam': 0.6, 'batt': 0.1, 'disp': 0.15},
        'hard_filters': {
            'score_cam': lambda x: x >= CAM_THRESHOLD,
            'video_resolution_rank': lambda x: x >= 4,
            'storage_gb': lambda x: x >= 256.0
        }
    },
    'Pin_Trau_Sac_Nhanh': {
        'weights': {'perf': 0.1, 'cam': 0.1, 'batt': 0.7, 'disp': 0.1},
        'hard_filters': {
            'battery_mah': lambda x: x >= 6000,
            'fast_charge_w': lambda x: x >= 33,
        }
    },

    'Nho_Gon_Nhe_Tay': {
        'weights': {'perf': 0.2, 'cam': 0.3, 'batt': 0.2, 'disp': 0.3},
        # ĐIỀU KIỆN CỨNG: Bắt buộc trọng lượng phải thuộc nhóm 1 (Dưới 185g)
        'hard_filters': {'weight_tier': lambda x: x == 1} 
    },

    'Khang_Nuoc': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {'ip_certified': lambda x: x == 1}
    },

    'Bat_Ky': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {}
    }
}


# Test

In [ ]:
engine = KBRSEngine(df_ready_for_kbrs, MAPPING_MATRIX_CONFIG)

In [64]:
# ==========================================
# 4. CHẠY TEST CÁC KỊCH BẢN THỰC TẾ

print("==== KỊCH BẢN 1: KHÁCH HÀNG QUAY VLOG, NGÂN SÁCH DƯỚI 15 TRIỆU ====")
display(engine.recommend(user_needs='Chup_Hinh_Quay_Phim', max_budget=15000000))
print("\n")

print("==== KỊCH BẢN 2: KHÁCH HÀNG MUA MÁY CHO PHỤ HUYNH (LIÊN LẠC CƠ BẢN) ====")
# Không có điều kiện cứng, ưu tiên pin (0.7), các điểm khác thấp
display(engine.recommend(user_needs='Lien_Lac_Co_Ban', max_budget=4000000))
print("\n")

print("==== KỊCH BẢN 3: TÌM FLAGSHIP TỐT NHẤT ====")
# Ép price_segment == 'Flagship', trọng số cân bằng
display(engine.recommend(user_needs='Flagship'))

==== KỊCH BẢN 1: KHÁCH HÀNG QUAY VLOG, NGÂN SÁCH DƯỚI 15 TRIỆU ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
110,Xiaomi Redmi Note 13 Pro Plus 5G 8GB 256GB,6490000,3.64,6.18,7.93,6.29,5.99
81,Redmi Note 13 Pro 5G 12GB 512GB,6990000,4.59,6.18,5.59,6.29,5.90
44,Xiaomi Redmi Note 15 Pro 5G 12GB 256GB,10890000,4.44,6.11,5.51,6.29,5.83




==== KỊCH BẢN 2: KHÁCH HÀNG MUA MÁY CHO PHỤ HUYNH (LIÊN LẠC CƠ BẢN) ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
379,Xiaomi Redmi Note 13 6GB 128GB,3790000,2.41,3.67,3.98,6.29,4.02
152,Itel RS4 12GB 256GB NFC,3390000,3.98,1.76,4.52,2.29,3.97
362,Itel RS4 8GB 256GB NFC,3290000,3.12,1.76,4.52,2.29,3.88




==== KỊCH BẢN 3: TÌM FLAGSHIP TỐT NHẤT ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
170,OPPO Find N6 16GB 512GB,63990000,8.49,7.03,6.74,8.29,7.82
262,OPPO Find X9 Ultra 12GB 512GB,48990000,7.63,9.03,6.03,6.29,7.49
267,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,5.63,7.68,6.29,6.88


In [69]:
print("==== TÌM MÁY VỪA CHƠI GAME VỪA PIN TRÂU ====")
# Hệ thống sẽ yêu cầu máy đó phải có: is_heavy_gaming == True VÀ battery_beast == True
display(engine.recommend(user_needs=['Choi_Game', 'Pin_Trau_Sac_Nhanh']))

print("==== MÁY NHẸ, FLAGSHIP ====")
display(engine.recommend(user_needs=['Nho_Gon_Nhe_Tay','Flagship']))


print("\n==== TÌM MÁY CHƠI GAME NHƯNG PHẢI QUAY VLOG MƯỢT (NGÂN SÁCH 35TR) ====")
display(engine.recommend(user_needs=['Choi_Game', 'Chup_Hinh_Quay_Phim'], max_budget=35000000))


print("\n==== TEST LỖI XUNG ĐỘT LOGIC CỦA NGƯỜI DÙNG ====")
# Khách hàng đòi Flagship cao cấp nhưng chọn nhu cầu Giá Rẻ
display(engine.recommend(user_needs=['Flagship', 'Gia_Re'])) 
# Kết quả sẽ in ra: "Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI..."

==== TÌM MÁY VỪA CHƠI GAME VỪA PIN TRÂU ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
267,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,5.63,7.68,6.29,7.69
170,OPPO Find N6 16GB 512GB,63990000,8.49,7.03,6.74,8.29,7.62
196,OPPO Find X9 16GB 512GB,25990000,8.44,4.41,7.38,6.29,7.47


==== MÁY NHẸ, FLAGSHIP ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
262,OPPO Find X9 Ultra 12GB 512GB,48990000,7.63,9.03,6.03,6.29,7.41
4,Samsung Galaxy S20,21490000,3.51,2.91,2.99,4.29,3.49
376,iPhone 15 256GB | Chính hãng VN/A,20790000,4.86,3.12,2.12,2.00,3.07



==== TÌM MÁY CHƠI GAME NHƯNG PHẢI QUAY VLOG MƯỢT (NGÂN SÁCH 35TR) ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
267,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,5.63,7.68,6.29,7.16
279,Xiaomi 15 Ultra 5G 16GB 1TB,28490000,8.68,5.24,6.82,6.29,7.03
319,Xiaomi 15 Ultra 5G 16GB 512GB,26490000,8.18,5.24,6.82,6.29,6.82



==== TEST LỖI XUNG ĐỘT LOGIC CỦA NGƯỜI DÙNG ====


'Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI tất cả các nhu cầu bắt buộc của bạn.'